# Tutorial 05: 2D Yukawa Potential, Observables, HDF5 Storage, and Animations

> **Language / Idioma:** [Español](../es/05_potencial_yukawa_y_observables.ipynb) | **English**

This notebook demonstrates a full scientific workflow with `Dirac-Solver`:
1. Setting up a 2D wavepacket scattering off a short-range `YukawaPotential` ($V(r) = -V_0 e^{-a r}/r$).
2. Saving simulation snapshots and observables into a chunked HDF5 archive via `HDF5Storage`.
3. Visualizing 2D probability density $\rho(x, y)$ overlaid with the probability current vector field $\mathbf{J}(x, y)$.
4. Exporting a GIF animation using `create_animation`.

In [ ]:
import sys, os
sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from dirac_solver import (
    Grid, ConstantSpinor, GaussianPacket,
    DiracProblemBuilder, DiracSolver, YukawaPotential,
    Observables, HDF5Storage, _core
)
from dirac_solver.storage import create_animation

## 1. Simulation Configuration and Execution

In [ ]:
shape = (100, 100)
spacing = (0.1, 0.1)
grid = Grid(shape, spacing)

spinor = ConstantSpinor(momentum=[4.0, 1.0, 0.0], mass=1.0)
packet = GaussianPacket(spinor, center=[-2.5, -0.5, 0.0], spatial_width=1.0)
potential = YukawaPotential(strength=1.5, range=0.5)
boundary = _core.AbsorbingBoundary(strength=0.1, width=15)

problem = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(packet)
    .set_potential(potential)
    .set_boundary_condition(boundary)
    .set_time_parameters(time_step=0.002, total_time=4.0)
    .build()
)

output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)
hdf5_path = os.path.join(output_dir, 'yukawa_2d_en.hdf5')
gif_path = os.path.join(output_dir, 'yukawa_2d_en.gif')

solver = DiracSolver(problem)
obs = Observables(solver)
storage = HDF5Storage(hdf5_path, problem)

solver.run_simulation(storage_handler=storage, save_every_n_steps=25, observables_handler=obs)

## 2. 2D Probability Density and Current Vector Field $\mathbf{J}(x, y)$

In [ ]:
last_rho = obs.get_history('probability_density')[-1].reshape(shape)
last_J = obs.get_history('probability_current')[-1]
jx = last_J[:, 0].reshape(shape)
jy = last_J[:, 1].reshape(shape)

x = grid.coords[:, 0, 0]
y = grid.coords[0, :, 1]
X, Y = np.meshgrid(x, y, indexing='ij')

plt.figure(figsize=(9, 7))
plt.imshow(
    last_rho.T, origin='lower', cmap='viridis',
    extent=[x[0], x[-1], y[0], y[-1]]
)
plt.colorbar(label='Probability Density $\\rho$')
skip = 5
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], jx[::skip, ::skip], jy[::skip, ::skip], color='white')
plt.xlabel('Position $x$')
plt.ylabel('Position $y$')
plt.title('2D Yukawa Scattering: Probability Density & Current')
plt.show()

## 3. Generating the GIF Animation

In [ ]:
create_animation(hdf5_path, gif_path, interval=100)
print(f'Animation saved to: {gif_path}')